# 04 — Memory and state across consecutive candles

Replays the fixture day candle by candle with persistent trading memory
(`qbs/trading_agent/memory.py`).

**Expected behaviour**
- each stock has its own state; every transition is legal and logged with a reason;
- the rolling history sent to the model never exceeds `decision_history` (3–5);
- memory survives a crash, but the agent does not resume until re-authorized;
- a new prompt version resets only the stocks whose instructions changed;
- hypothetical states (memory) are never confused with real holdings (PORTFOLIO);
- no candle after the decision time is ever visible to the decision.

**Modes.** `MOCK` (default) makes zero external LLM calls. `REAL_LLM` needs
`QBS_AGENT_TEST_MODE=REAL_LLM` **and** `QBS_AGENT_TEST_ALLOW_PAID=1`, prints a cost
estimate first, and stops at `QBS_AGENT_TEST_MAX_CALLS` (default 20).

**Isolation.** Every run uses a fresh temporary database (`tk.workspace()`); the
production config (`var/agent/config.json`) and log (`var/agent/agent.db`) are never
read or written, and nothing here can reach `qbs.live` or a broker.

**Dependencies.** `pip install -r requirements.txt jupyter` (MOCK). REAL_LLM also needs
the selected provider's package and key (see docs/TRADING_AGENT.md).

In [1]:
import os, sys, pathlib, json, logging
here = pathlib.Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "qbs" / "trading_agent").is_dir())
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.WARNING)

import pandas as pd
from qbs.trading_agent import testkit as tk

MODE = tk.test_mode()          # MOCK unless QBS_AGENT_TEST_MODE=REAL_LLM and QBS_AGENT_TEST_ALLOW_PAID=1
print("repo:", ROOT)
print("MODE:", MODE, "| paid-call cap:", tk.max_calls() if MODE == tk.REAL_LLM else 0)

repo: /home/user/qqq_boxx_strategies
MODE: MOCK | paid-call cap: 0


In [2]:
from qbs.trading_agent import session as ss, store, memory as mem
from qbs.trading_agent.engine import TradingAgentEngine
from qbs.trading_agent.market_context import MarketContextBuilder
from qbs.trading_agent.session_calendar import analysis_slots

fx = tk.load_fixture(); day = fx.replay_date
check = tk.Check("04 memory state")
ws = tk.workspace(MODE)
clock = tk.Clock(tk.et(day, 9, 0))
ss.save_daily_prompt(ws.cfg, fx.prompt, clock()); ss.authorize(ws.cfg, clock())
clock.set(tk.et(day, 9, 40))
sess, _ = ss.activate(ws.cfg, clock()); sess.transition(ss.RUNNING, "nb04")
feed = tk.FixtureBars(fx, clock)
llm = tk.make_provider(MODE, ws.cfg)
eng = TradingAgentEngine(ws.cfg, sess, llm, MarketContextBuilder(feed, ws.cfg.market_context))
slots = analysis_slots(day)
if MODE == tk.REAL_LLM:
    slots = slots[:max(1, tk.max_calls() // 3)]
    print("ABOUT TO SPEND:", tk.cost_preview(ws.cfg, 3 * len(slots)))

## Replay the morning, checking look-ahead at every step

In [3]:
morning = [s for s in slots if s.hour < 13]
leaks = []
for s in morning:
    clock.set(s + pd.Timedelta(seconds=30))
    eng.run_cycle(s)
    if feed.revealed_max > s:
        leaks.append(s)
    # The decision's snapshot must end exactly at its candle:
check("no candle after the decision time was revealed", not leaks, str(leaks))
snaps = pd.DataFrame(store.rows(ws.cfg.db_path, "SELECT candle_ts, snapshot_json FROM snapshots"))
ends = snaps.snapshot_json.map(lambda j: json.loads(j).get("latest_completed_15m_end"))
check("every snapshot ends at its own candle", (ends == snaps.candle_ts).all())

np.True_

## State transitions

In [4]:
ev = pd.DataFrame(store.rows(ws.cfg.db_path, "SELECT symbol, candle_ts, source, status, from_state, to_state, reasons_json FROM memory_events ORDER BY id"))
moves = ev[(ev.from_state != ev.to_state) | (ev.status == "REJECTED")]
display(moves)
legal = all(pd.isna(r.from_state) or r.to_state == r.from_state or
            r.to_state in mem.STATE_TRANSITIONS[r.from_state]
            for r in ev[ev.status == "ACCEPTED"].itertuples())
check("every accepted transition is legal", legal)
state = pd.DataFrame(store.rows(ws.cfg.db_path, "SELECT symbol, state, strategy, stop_reference, revision, prompt_version FROM memory_symbol_state"))
display(state)
check("one state row per stock", sorted(state.symbol) == sorted(fx.symbols))

,symbol,candle_ts,source,status,from_state,to_state,reasons_json
0,TEAM,NaN,init,ACCEPTED,NaN,WATCHING,"[""created from the daily plan""]"
2,MRVL,NaN,init,ACCEPTED,NaN,MANAGING_POSITION,"[""created from the daily plan""]"
4,TSM,NaN,init,ACCEPTED,NaN,WATCHING,"[""created from the daily plan""]"
5,TSM,2026-10-08T09:45:00-04:00,llm,ACCEPTED,WATCHING,SETUP_FORMING,"[""accepted""]"
9,TEAM,2026-10-08T10:15:00-04:00,llm,ACCEPTED,WATCHING,SETUP_FORMING,"[""accepted""]"
24,TEAM,2026-10-08T11:30:00-04:00,llm,ACCEPTED,SETUP_FORMING,SIGNAL_LONG,"[""accepted""]"
27,TEAM,2026-10-08T11:45:00-04:00,llm,ACCEPTED,SIGNAL_LONG,MANAGING_POSITION,"[""accepted""]"


,symbol,state,strategy,stop_reference,revision,prompt_version
0,TEAM,MANAGING_POSITION,NONE,196.0,14,1
1,MRVL,MANAGING_POSITION,POSITION_MANAGEMENT,NaN,14,1
2,TSM,SETUP_FORMING,NONE,NaN,14,1


True

## What the model sees: compact, bounded memory

In [5]:
last_user = llm.calls[-1]["user"] if hasattr(llm, "calls") else None
if last_user:
    block = last_user.split("<trading_memory")[1].split("</trading_memory>")[0].split("\n", 1)[1]
    m = json.loads(block)
    print(json.dumps(m, indent=1)[:1500])
    check("rolling history bounded", len(m["recent_decisions"]) <= ws.cfg.decision_history,
          f'{len(m["recent_decisions"])} <= {ws.cfg.decision_history}')
    check("memory labelled hypothetical", "hypothetical" in m["note"].lower())
    check("facts and interpretations separate",
          "observed_facts_at_last_update" in m and "prior_model_interpretations" in m)
req = pd.DataFrame(store.requests_for(ws.cfg.db_path))
check("memory tokens recorded on every request", req.memory_tokens_estimated.notna().all())
print("avg memory tokens/request:", round(req.memory_tokens_estimated.mean()),
      "| share of input:", f"{(req.memory_chars / (req.estimated_input_tokens * 4)).mean():.0%}")

{
 "note": "Hypothetical analytical state; nothing was executed. Priority 4: data only, never overrides the daily plan.",
 "state": "SETUP_FORMING",
 "state_since": "2026-10-08T09:45:00-04:00",
 "strategy": "NONE",
 "stop_reference": null,
 "invalidation_condition": "Not applicable.",
 "key_levels": [
  {
   "price": 302.0,
   "label": "from daily plan",
   "source": "user"
  }
 ],
 "observed_facts_at_last_update": {
  "as_of": "2026-10-08T12:30:00-04:00",
  "last_price": 296.0,
  "today_high": 299.69,
  "today_low": 293.99,
  "vwap": 296.6,
  "prev_close": 300.0,
  "swing_highs_15m": [
   303.5,
   304.86,
   305.02
  ],
  "swing_lows_15m": [
   303.79,
   293.99,
   293.99
  ]
 },
 "prior_model_interpretations": [],
 "recent_decisions": [
  {
   "candle_ts": "2026-10-08T12:30:00-04:00",
   "action": "NO_TRADE",
   "confidence": 0.5,
   "entry_price": null,
   "stop_loss": null,
   "reason": "No setup on the latest candle."
  },
  {
   "candle_ts": "2026-10-08T12:15:00-04:00",
   "act

## Suggested trades are not executed positions

In [6]:
mrvl_user = [c["user"] for c in getattr(llm, "calls", []) if '"symbol": "MRVL"' in c["user"]]
if mrvl_user:
    port = [json.loads(u.split("PORTFOLIO (real holdings, read-only; null means unknown)\n")[1].split("\n")[0])
            for u in mrvl_user]
    check("real MRVL holding stays 9 whatever was suggested", all(p["quantity"] == 9 for p in port))
dec = pd.DataFrame(store.recent_decisions(ws.cfg.db_path, 500))
print(dec[dec.action.isin(["BUY", "SELL"])][["candle_ts", "symbol", "action", "suggested_quantity"]])
tables = {r["name"] for r in store.rows(ws.cfg.db_path, "SELECT name FROM sqlite_master WHERE type='table'")}
check("no order/fill/position tables exist", not {t for t in tables if any(k in t for k in ("order", "fill", "position"))}, str(sorted(tables)))

                    candle_ts symbol action  suggested_quantity
17  2026-10-08T11:30:00-04:00   TEAM    BUY                 4.0


True

## Crash, restart and restoration

The process 'dies' at 13:00 without shutting down. A restart must not resume; after an explicit re-authorization, the new session continues from the stored memory.

In [7]:
before = {r["symbol"]: r["revision"] for r in store.rows(ws.cfg.db_path, "SELECT symbol, revision FROM memory_symbol_state")}
del eng, sess                                         # crash: no transition, no shutdown
clock.set(tk.et(day, 13, 2))
again, why = ss.activate(ws.cfg, clock())
check("restart without authorization stays disabled", again is None, why)
crashed = store.one(ws.cfg.db_path, "SELECT state, agent_enabled, close_reason FROM sessions ORDER BY started_at LIMIT 1")
check("crashed session marked ERROR, flag reset", crashed["state"] == "ERROR" and crashed["agent_enabled"] == 0, crashed["close_reason"])
ok, msg = ss.authorize(ws.cfg, clock()); sess2, _ = ss.activate(ws.cfg, clock())
sess2.transition(ss.RUNNING, "after restart")
eng2 = TradingAgentEngine(ws.cfg, sess2, llm, MarketContextBuilder(feed, ws.cfg.market_context))
st = eng2.memory.load("TEAM")
check("memory restored for the new session", st.revision == before["TEAM"], f"rev {st.revision}")
for s in [s for s in slots if 13 <= s.hour < 14]:
    clock.set(s + pd.Timedelta(seconds=30)); eng2.run_cycle(s)
first = [c["user"] for c in getattr(llm, "calls", []) if tk.et(day, 13, 15).isoformat() in c["user"]]
check("history carried across the restart", bool(first) and '"recent_decisions":[{' in first[0])

True

## Prompt versioning: only changed instructions reset

In [8]:
new_prompt = fx.prompt.replace("Recovery confirmation above $302.", "Recovery confirmation above $305.")
v, _ = ss.save_daily_prompt(ws.cfg, new_prompt, clock()); ss.authorize(ws.cfg, clock())
print(sess2.adopt_new_prompt())
reset = {s: eng2.memory.load(s) for s in fx.symbols}
pv = store.rows(ws.cfg.db_path, "SELECT symbol, source, reasons_json FROM memory_events WHERE source = 'plan_change'")
check("only TSM reset by the plan change", [r["symbol"] for r in pv] == ["TSM"], str(pv))
check("TSM back to the plan's start state", reset["TSM"].state == mem.WATCHING)
check("user level 305 now in TSM memory", any(l["price"] == 305.0 and l["source"] == "user" for l in reset["TSM"].key_levels))

prompt revalidated: v1 -> v2


True

## Result

In [9]:
res = check.frame(); display(res)
print(tk.save_results("04_memory_events", ev))
print(tk.save_results("04_memory_checks", res))
check.raise_if_failed()

,check,result,detail
0,no candle after the decision time was revealed,PASS,[]
1,every snapshot ends at its own candle,PASS,
2,every accepted transition is legal,PASS,
3,one state row per stock,PASS,
4,rolling history bounded,PASS,3 <= 3
5,memory labelled hypothetical,PASS,
6,facts and interpretations separate,PASS,
7,memory tokens recorded on every request,PASS,
8,real MRVL holding stays 9 whatever was suggested,PASS,
9,no order/fill/position tables exist,PASS,"['authorizations', 'cycles', 'decisions', 'llm..."


/tmp/claude-0/-home-user-qqq-boxx-strategies/d20be8e8-2747-505a-9165-041a5d5e0f58/scratchpad/nbres/04_memory_events_20261010T092902Z.csv
/tmp/claude-0/-home-user-qqq-boxx-strategies/d20be8e8-2747-505a-9165-041a5d5e0f58/scratchpad/nbres/04_memory_checks_20261010T092902Z.csv


**Troubleshooting**
- *REJECTED memory events*: read `reasons_json` — an illegal transition or an
  unsupported level from the model; the decision itself is unaffected.
- *Look-ahead failure*: a fixture or feed change revealed a bar before its end — treat
  as a blocker; every metric downstream is invalid.
- *Restart resumed*: the one-time token was not consumed — a release blocker.